# 03 · Normalization — per-capita & per-calorie

**Question:** absolute totals favour big countries. Per-capita and per-calorie reshuffle the story and are where non-obvious findings hide. Needs extra data (degrades gracefully if absent).

**Download to enable:** `Population.csv` (Population > Annual population, Total both sexes) and `Food_Balances.csv` (Food Balances > kcal/capita/day).

## Setup

In [1]:
import sys; sys.path.append('..')
import pandas as pd, matplotlib.pyplot as plt
from src import load, clean, footprint, viz

data = load.load_all()
emis, inten, land = data['emissions'], data['intensities'], data['landuse']
emis_c, _  = clean.split_countries_aggregates(emis)
inten_c, _ = clean.split_countries_aggregates(inten)
land_c, _  = clean.split_countries_aggregates(land)
YEAR = clean.pick_analysis_year(emis_c)
ELEM = clean.pick_co2eq_element(emis_c)
print('year', YEAR, '| element', ELEM)

year 2023 | element Emissions (CO2eq) (AR5)


## Per-capita emissions

In [2]:
pop_df = load.try_read('population')
if pop_df is not None:
    pop_c, _ = clean.split_countries_aggregates(pop_df)
    pop = footprint.population_series(pop_c, YEAR)
    emis_by_country = footprint.rank_countries(emis_c, YEAR, ELEM)
    pc = footprint.per_capita(emis_by_country, pop)
    print('Population unit is often "1000 persons" — per_capita is emissions per 1000 people; scale as needed.')
    viz.barh_ranking(pc['per_capita'], f'Agrifood emissions per capita — {YEAR}', 'CO2eq per (1000) people', n=15, color='teal'); plt.show()
    display(pc.head(15))
    print('\nContrast: absolute top-5 vs per-capita top-5')
    print('absolute:', list(emis_by_country.head(5).index))
    print('per-capita:', list(pc.head(5).index))
else:
    print('Download Population.csv to run per-capita.')

[optional] Population.csv not found in data/raw/ — download it to enable this analysis. Skipping.
Download Population.csv to run per-capita.


> **The interesting bit** is the *contrast* line: if the per-capita leaderboard is completely different from the absolute one, that divergence is a finding in itself.

## Per-calorie intensity (needs Food Balances)

In [3]:
fb = load.try_read('food_balances')
if fb is not None:
    fb_c, _ = clean.split_countries_aggregates(fb)
    kcal_elem = [e for e in fb_c['Element'].unique() if 'kcal' in str(e).lower()]
    print('kcal elements available:', kcal_elem[:5])
    print('\nNext step: join commodity intensity (kg CO2eq/kg) to kcal/kg to get\n'
          'CO2eq per calorie — reframes "is beef efficient per unit of nutrition".')
else:
    print('Download Food_Balances.csv to run per-calorie.')

[optional] Food_Balances.csv not found in data/raw/ — download it to enable this analysis. Skipping.
Download Food_Balances.csv to run per-calorie.
